# Reddit Live Sentiment — Interactive Dashboard

Reads the live comment buffer that the streaming notebook writes to MongoDB and presents
it through interactive `ipywidgets` views: per-subreddit sentiment, cross-genre
comparison, word clouds, and a Drake-vs-Kendrick artist breakdown.

Run `streaming_ingestion.ipynb` first (and leave it running) so there is live data in
MongoDB for this dashboard to read.

**Note on running:** requires a running local MongoDB instance.

## Setup

Install dashboard dependencies and import the scoring tools.

In [ ]:
import sys
!{sys.executable} -m pip install ipywidgets wordcloud

import nltk
nltk.download("vader_lexicon")

from ipywidgets import interact, widgets
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import pandas as pd
from pymongo import MongoClient
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
from IPython.display import clear_output

## 1. Connect and define scorers

Connect to the same MongoDB collection the streamer writes to, and define the VADER and
TextBlob scoring helpers.

In [ ]:
client = MongoClient("mongodb://localhost:27017/")
col = client["reddit_sentiment"]["reddit_comments"]

sia = SentimentIntensityAnalyzer()

def vader(text):
    return sia.polarity_scores(text)["compound"] if text else 0

def tb(text):
    return TextBlob(text).sentiment.polarity if text else 0

## 2. Load and enrich live data

`load_live_df()` reads all buffered comments, scores them with both models, and tags each
with an artist using subreddit signal first (e.g. r/drizzy → Drake) then keyword fallback.

In [ ]:
def load_live_df():
    docs = list(col.find().sort("published_at", 1))

    if not docs:
        return pd.DataFrame()

    df = pd.DataFrame(docs)
    df["published_at"] = pd.to_datetime(df["published_at"], utc=True)

    # compute sentiment
    df["vader"] = df["text"].apply(vader)
    df["textblob"] = df["text"].apply(tb)
    
    df["artist"] = df.apply(lambda row: detect_artist(row["text"], row["subreddit"]), axis=1)

    return df

import re

def detect_artist(text, subreddit):

    t = (text or "").lower()
    s = (subreddit or "").lower()

    # Subreddit-based detection (strong signal)
    if s == "drizzy":
        return "Drake"
    if s == "kendricklamar":
        return "Kendrick Lamar"

    # Text-based detection (fallback)
    if re.search(r"\b(drake|champagne papi|ovo)\b", t):
        return "Drake"

    if re.search(r"\b(kendrick|k-dot|kdot|kung fu kenny|kendrick lamar)\b", t):
        return "Kendrick Lamar"

    return "Unknown"

## 3. Per-subreddit live sentiment

Interactive rolling-average sentiment for a chosen subreddit and model.

In [ ]:
subreddits = ["hiphopheads", "indieheads", "popheads"]
models = ["VADER", "TextBlob"]

def live_sentiment(subreddit="hiphopheads", model="VADER"):
    df = load_live_df()

    if df.empty:
        print("No live data yet...")
        return

    df_sub = df[df["subreddit"] == subreddit]

    if df_sub.empty:
        print(f"No data yet for {subreddit}")
        return

    col_name = "vader" if model == "VADER" else "textblob"

    plt.figure(figsize=(10, 4))
    plt.plot(df_sub["published_at"], df_sub[col_name].rolling(10, min_periods=1).mean(),
             label=f"{subreddit} ({model})", linewidth=2)

    plt.ylim(-1, 1)
    plt.title(f"Live Sentiment Rolling Average — {subreddit}")
    plt.grid(True)
    plt.legend()
    plt.show()

interact(
    live_sentiment,
    subreddit=widgets.Dropdown(options=subreddits, description="Subreddit:"),
    model=widgets.ToggleButtons(options=models, description="Model:")
)

## 4. Cross-genre comparison

Rolling sentiment for all three genre subreddits on one axis.

In [ ]:
def compare_live_genres(model="VADER"):
    df = load_live_df()

    if df.empty:
        print("No live data yet...")
        return

    col_name = "vader" if model == "VADER" else "textblob"

    plt.figure(figsize=(10, 5))

    for sub in subreddits:
        d = df[df["subreddit"] == sub][col_name]
        if len(d) > 0:
            plt.plot(
                d.rolling(10, min_periods=1).mean().values,
                label=sub
            )

    plt.ylim(-1, 1)
    plt.title(f"Live Sentiment Comparison Across Genres ({model})")
    plt.legend()
    plt.grid(True)
    plt.show()

interact(
    compare_live_genres,
    model=widgets.ToggleButtons(options=models, description="Model:")
)

## 5. Per-subreddit word cloud

Most frequent terms in the live comments for a chosen subreddit.

In [ ]:
def live_wordcloud(subreddit="hiphopheads"):
    df = load_live_df()

    if df.empty:
        print("No live comments yet...")
        return

    df_sub = df[df["subreddit"] == subreddit]

    if df_sub.empty:
        print(f"No comments yet for {subreddit}")
        return

    text = " ".join(df_sub["text"].astype(str).tolist()).lower()

    wc = WordCloud(width=800, height=400, background_color="white").generate(text)

    plt.figure(figsize=(12, 6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Live Word Cloud — {subreddit}")
    plt.show()

interact(
    live_wordcloud,
    subreddit=widgets.Dropdown(options=subreddits, description="Subreddit:")
)

## 6. Artist views: Drake vs Kendrick Lamar

Real-time rolling sentiment, average sentiment, and word clouds for each artist.

In [ ]:
def artist_sentiment(model="VADER"):
    df = load_live_df()

    if df.empty:
        print("No data yet...")
        return

    col_name = "vader" if model == "VADER" else "textblob"

    df_d = df[df["artist"] == "Drake"]
    df_k = df[df["artist"] == "Kendrick Lamar"]

    if df_d.empty and df_k.empty:
        print("No artist-related comments detected yet.")
        return

    plt.figure(figsize=(10, 5))

    if not df_d.empty:
        plt.plot(
            df_d[col_name].rolling(10, min_periods=1).mean().values,
            label="Drake",
            linewidth=2,
            color="green"
        )

    if not df_k.empty:
        plt.plot(
            df_k[col_name].rolling(10, min_periods=1).mean().values,
            label="Kendrick Lamar",
            linewidth=2,
            color="purple"
        )

    plt.axhline(0, color="black", linewidth=1)
    plt.ylim(-1, 1)
    plt.title(f"Real-Time Sentiment: Drake vs Kendrick ({model})")
    plt.grid(True)
    plt.legend()
    plt.show()

interact(
    artist_sentiment,
    model=widgets.ToggleButtons(options=models, description="Model:")
)

In [ ]:
def artist_avg_sentiment(model="VADER"):
    df = load_live_df()

    if df.empty:
        print("No data yet...")
        return

    col_name = "vader" if model == "VADER" else "textblob"

    avg_df = df.groupby("artist")[col_name].mean().reset_index()

    # Keep only real artists
    avg_df = avg_df[avg_df["artist"].isin(["Drake", "Kendrick Lamar"])]

    if avg_df.empty:
        print("No artist comments detected yet.")
        return

    plt.figure(figsize=(8,4))
    plt.bar(avg_df["artist"], avg_df[col_name], color=["green", "purple"])
    plt.ylim(-1, 1)
    plt.axhline(0, color="black", linewidth=1)
    plt.title(f"Average Live Sentiment — Drake vs Kendrick ({model})")
    plt.ylabel("Sentiment Score (-1 to 1)")
    plt.show()

interact(
    artist_avg_sentiment,
    model=widgets.ToggleButtons(options=models, description="Model:")
)

In [ ]:
def artist_wordcloud(artist="Drake"):
    df = load_live_df()

    if df.empty:
        print("No live data yet...")
        return

    df_a = df[df["artist"] == artist]

    if df_a.empty:
        print(f"No comments for {artist} yet.")
        return

    text = " ".join(df_a["text"].astype(str).tolist())

    wc = WordCloud(width=800, height=400, background_color="white").generate(text)

    plt.figure(figsize=(12,6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Live Word Cloud — {artist}")
    plt.show()

interact(
    artist_wordcloud,
    artist=widgets.Dropdown(options=["Drake", "Kendrick Lamar"], description="Artist:")
)